In [1]:
import os
import pandas as pd
import re
from datetime import datetime
import gc
import numpy as np


In [2]:
# Path to the current folder containing
folder_path = os.getcwd() 

# Get a list of all CSV files in the folder
file_list = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')], key=str.casefold)
print("\n".join(file_list))

october-2021-About.csv
october-2021-Complexity_of_the_crisis.csv
october-2021-Conditions_of_people_affected.csv
october-2021-Core_Indicators.csv
october-2021-Country_Indicator_Data.csv
october-2021-Crisis_Indicator_Data.csv
october-2021-Crisis_info.csv
october-2021-Data_Reliability.csv
october-2021-Impact_of_the_crisis.csv
october-2021-Imputed_and_missing_data_hidden.csv
october-2021-Indicator_Date_hidden.csv
october-2021-Indicator_Date_hidden2.csv
october-2021-Indicator_Metadata.csv
october-2021-INFORM_Severity_all_crises.csv
october-2021-INFORM_Severity_country.csv
october-2021-INFORM_Severity_hidden.csv
october-2021-Lists.csv
october-2021-Regional_Crises.csv
october-2021-Reliability.csv
october-2021-Reliability_updated.csv
october-2021-Trends.csv


In [3]:
# Initialize the variables
current_file_index = -1
release_date = None
month = None
year = None
year_month = None

Functions:

In [4]:
import os
import pandas as pd

def load_next_csv(file_list, folder_path, current_file_index):
    """
    Loads the next CSV file from the file list into a DataFrame.
    
    Parameters:
    file_list (list): List of CSV filenames to load.
    folder_path (str): Path to the folder containing the CSV files.
    current_file_index (int): Index of the current file to load.
    
    Returns:
    tuple: DataFrame loaded from the next CSV file, updated file index
    """
    # Check if there are more files to process
    if current_file_index >= len(file_list) - 1:
        print("No more CSV files left to load.")
        return None, current_file_index
    
    # Increment the file index to load the next file
    current_file_index += 1
    
    # Get the next file in the list
    current_file = file_list[current_file_index]
    file_path = os.path.join(folder_path, current_file)
    print(f"Opening file: {current_file}")
    
    # Load the CSV file into a DataFrame without headers for initial inspection
    df = pd.read_csv(file_path, header=None)
    
    return df, current_file_index


In [5]:
def drop_nan(df, column_name):
    """
    Cleans the DataFrame by removing rows where the specified column has NaN,
    removing columns with NaN in the header, and renaming unnamed columns.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    column_name (str): The name of the column to check for NaN values in rows.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame.
    """
    # Track rows with NaN in the specified column
    rows_to_delete = df[df[column_name].isna()].index.tolist()

    # Drop rows with NaN in the specified column
    df = df.dropna(subset=[column_name])

    # Track columns with NaN in the header
    columns_to_delete = [col for col in df.columns if pd.isna(col)]
    
    # Replace NaN headers with placeholders (e.g., "Unnamed")
    df.columns = [col if pd.notna(col) else f"Unnamed_{i}" for i, col in enumerate(df.columns)]
    
    # Optionally drop columns that were NaN if not needed
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    # Print deleted rows and columns
    print("Deleted rows (indices):", rows_to_delete)
    print("Deleted columns:", columns_to_delete)

    return df


In [6]:
def clean_dataframe(df, crisis_id_col):
    """
    Cleans the DataFrame by dropping rows where the Crisis ID column has NaN,
    resetting the index, and setting row 0 as the header.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to clean.
    crisis_id_col (str): The name of the Crisis ID column to check for NaN values.
    
    Returns:
    pd.DataFrame: The cleaned DataFrame with updated headers.
    """
    # List to collect rows to drop
    rows_to_drop = []
    cols_to_drop = []
    
    # Identify rows where the Crisis ID column has NaN
    for row in range(len(df)):
        if pd.isna(df.iloc[row, crisis_id_col]):
            rows_to_drop.append(row)
    
    # Drop rows with NaN in the Crisis ID column
    print(f"Dropping NaN rows: {rows_to_drop}")
    df = df.drop(index=rows_to_drop)

    # Reset the index
    df = df.reset_index(drop=True)

    # Identify columns where header is NaN
    for col in range(0, len(df.columns)):
        if pd.isna(df.iloc[0, col]):
            cols_to_drop.append(df.columns[col])

    # Drop columns with NaN in the header
    print(f"Dropping NaN columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

    # Make row 0 the header
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])

    # Rename columns for consistency
    column_mappings = {
        'Crisis Id': 'Crisis Id',
        'Crisisid': 'Crisis Id',
        'CrisisID': 'Crisis Id',
        'ISO3 code': 'ISO3',
        'Iso3 Code': 'ISO3',
        'Iso3': 'ISO3',
        'Iso 3': 'ISO3',
        # Add any other variations that need standardization
    }
    df = df.rename(columns=column_mappings)

    # Standardize target columns to titlecase strings and strip whitespace
    target_columns = ['Crisis', 'Drivers', 'Crisis Id', 'Country', 'ISO3']
    for col in target_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.title().str.strip()

    df.columns = df.columns.str.title()

    return df


In [7]:
def get_dataframe_stats(df):
    """
    Prints the number of rows and columns in the DataFrame.
    
    Parameters:
    df (pd.DataFrame): The DataFrame to analyze.
    """
    num_rows, num_columns = df.shape
    return f"{num_rows} rows x {num_columns} columns"

In [8]:
#####-MERGE-#####


def merge_df(df_1, df_2, on_col, merge_type='outer'):
    # Add suffixes to all columns except the key column
    df_1 = df_1.rename(columns={col: f"{col}[df1]" for col in df_1.columns if col != on_col})
    df_2 = df_2.rename(columns={col: f"{col}[df2]" for col in df_2.columns if col != on_col})
    
    # Merge the DataFrames
    merged_df = pd.merge(df_1, df_2, on=on_col, how=merge_type)

    # Remove suffixes by renaming columns
    merged_df.columns = [col.replace('[df1]', '').replace('[df2]', '') for col in merged_df.columns]
    
    # Identify duplicated columns (keeping only the first occurrence)
    duplicate_columns = [col for col in merged_df.columns if merged_df.columns.tolist().count(col) > 1]

    # Remove duplicate columns, keeping only the first occurrence
    merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

    # Identify rows where all values are duplicates
    duplicate_rows = merged_df.duplicated(keep=False)  # Marks all rows that are full duplicates

    # Get the indices of duplicate rows
    duplicate_indices = merged_df[duplicate_rows].index.tolist()

    # Count the number of duplicate rows
    num_deleted_rows = len(duplicate_indices)

    # Drop these duplicate rows from the DataFrame
    merged_df_cleaned = merged_df[~duplicate_rows].reset_index(drop=True)

    # Print the number of deleted rows and the row indices
    print(f"Number of rows deleted: {num_deleted_rows}")
    print("Indices of deleted duplicate rows:", duplicate_indices)
    print("DataFrame after removing fully duplicate rows:")
    print(merged_df_cleaned)

    return merged_df_cleaned



About.csv

In [9]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-About.csv
Current file index is: 0


,0
0,NaN
1,release:
2,31/10/2021
3,INFORM SEVERITY INDEX
4,October 2021
5,OBJECTIVES AND PROCESS
6,The INFORM Severity Index summarises a wide ra...
7,ANALYTICAL FRAMEWORK AND METHODOLOGY
8,The INFORM Severity Index is a composite indic...
9,NaN


In [10]:
# Function to check if a string is a date in different formats
def is_date(string):
    formats = ["%d/%m/%Y", "%Y-%m-%d %H:%M:%S"]
    for fmt in formats:
        try:
            return datetime.strptime(string, fmt).strftime("%d/%m/%Y")
        except ValueError:
            continue
    return False

In [11]:
current_file = file_list[current_file_index]
print(f"Current file: {current_file}")
if current_file.endswith("About.csv"):
    # Identify the first column by its index position
    first_column = df.columns[0]
    
    # Search for the word 'release' in the first column
    for i, row in df.iterrows():
        if str(row[first_column]).strip().lower() == "release:":
            # Check if the next row exists and if it contains a date
            next_row_value = df.at[i + 1, first_column] if i + 1 < len(df) else None
            formatted_date = is_date(str(next_row_value))
            if formatted_date:
                release_date = formatted_date
                day, month, year = release_date.split("/")  # Split the date into parts
                month = int(month)
                year = int(year)
                year_month = f"{year}_{month:02d}"
            break

Current file: october-2021-About.csv


In [12]:
# Display the extracted release date, month, year, and year_month
if release_date:
    print(f"Release date for {current_file}: {release_date}")
    print(f"Month: {month}")
    print(f"Year: {year}")
    print(f"Year-Month: {year_month}")
else:
    print("\n" + f"No release date found in {current_file}.")

Release date for october-2021-About.csv: 31/10/2021
Month: 10
Year: 2021
Year-Month: 2021_10


In [13]:
pd.set_option('display.max_columns', None)


df1. Complexity_of_the_crisis.csv

In [14]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Complexity_of_the_crisis.csv
Current file index is: 1


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,Empowerment,BTI - Democracy Status,Trust in society,Ethnic Fractionalisation,size of excluded ethnic groups,Ethnic fractionalisation,Gender Inequality,Income Gini coefficient,Inequality,Social cohesion,Conflict Intensity,Total killed in all crisis,Safety and security,Corruption Perception,Rule of Law (WGI),Rule of Law (BTI),Freedom in the World,Rule of Law,Society and safety,# of different types of affected population gr...,Diversity of groups affected,Impediments to entry into country (bureaucrati...,Restriction of movement (impediments to freedo...,Interference into implementation of humanitari...,"Violence against personnel, facilities and assets",Access of Humanitarian Actors to Affected Popu...,Denial of existence of humanitarian needs or e...,Restriction and obstruction of access to servi...,Access of People in need to Aid,Ongoing insecurity/hostilities affecting human...,Presence of mines and improvised explosive dev...,Physical constraints in the environment (obsta...,Physical and Security Constraints,Humanitarian access,Operating environment,NaN
2,NaN,NaN,NaN,NaN,MAX,14,10,NaN,1,0.5,NaN,0.75,65,NaN,NaN,NaN,3000,NaN,100,2.5,10,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,MIN,0,0,NaN,0,0,NaN,0,25,NaN,NaN,NaN,1,NaN,0,-2.5,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CRISIS,TYPE,CRISIS ID,COUNTRY,Iso3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Complexity_of_the_crisis.csv"):
    if df.iloc[1, 5] == "Empowerment" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-39) down into row 4 (columns 5-39)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    # Delete empty rows
    df = clean_dataframe(df, crisis_id_col = 2)
    df1 = df
else: print("Wrong file")

Dropping NaN rows: [0, 1, 2, 3, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
Dropping NaN columns: [40]


In [16]:
# Display the updated DataFrame
df1

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Ethnic Fractionalisation,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment
1,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,3.6,3.4,3.5,3.7,0,1.85,3.8,x,3.8,3.0499999999999994,10,5,7.5,4.2,4.2,3.5,3.7,3.9,4.8,5,5,0,3,2,3,4,2,3,5,3,3,3,5,5,5
2,Nagorno-Karabakh Conflict in Armenia,Conflict,ARM002,Armenia,Arm,2.9,1.5,2.2,0.2,0.2,0.2,1.7,0.6,1.15,1.1833333333333333,6,0,3,2.9,2.6,2,2.4,2.5,2.2,2,2,0,0,0,0,0,0,0,0,0,2,0,2,1,1.5
3,Nagorno-Karabakh conflict in Azerbaijan,Conflict,AZE002,Azerbaijan,Aze,3.6,3.3,3.45,0.8,0.6,0.7,2.1,0.2,1.1500000000000001,1.7666666666666668,10,2.8,6.4,3.5,3.1,3.5,4.5,3.7,4,3,3,2,0,1,0,2,0,0,0,1,3,1,4,2,2.5
4,Complex in Burundi,Complex crisis,BDI001,Burundi,Bdi,2.1,3.2,2.6500000000000004,1.3,0,0.65,3.5,1.7,2.6,1.9666666666666668,6,3.4,4.7,4.1,3.9,3.8,4.4,4.1,3.6,5,5,0,2,0,0,2,0,0,0,0,0,3,3,2,3.5
5,Conflict in Burkina Faso,Conflict,BFA002,Burkina Faso,Bfa,1.1,1.9,1.5,2.8,0,1.4,4.1,1.3,2.6999999999999997,1.8666666666666665,10,4,7,3,2.9,2.9,2.2,2.8,3.9,4,4,0,2,2,0,2,0,3,3,2,0,3,4,3,3.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,Complex crisis in Venezuela,Complex crisis,VEN001,Venezuela,Ven,2.5,3.5,3,1.3,1.2,1.25,3.1,2.5,2.8,2.35,6,5,5.5,4.2,4.8,4.1,4.2,4.3,4.1,2,2,2,3,2,2,4,3,3,5,3,0,3,4,4,3
126,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,4.3,4.3,4.3,3.1,0,1.55,5,1.5,3.25,3.033333333333333,10,3.6,6.8,4.3,4.3,4.4,4.5,4.4,4.7,5,5,2,3,3,2,5,2,3,5,2,1,3,4,5,5
127,Mixed migration flows in Yemen,International displacement,YEM002,Yemen,Yem,4.3,4.3,4.3,3.1,0,1.55,5,1.5,3.25,3.033333333333333,10,3.6,6.8,4.3,4.3,4.4,4.5,4.4,4.7,2,2,2,3,3,2,5,2,2,4,2,1,3,4,4,3
128,Drought in Zambia,Drought,ZMB002,Zambia,Zmb,2.1,2.1,2.1,3.5,0,1.75,3.6,4,3.8,2.5500000000000003,0,1.6,0.8,3.3,3,2.9,2.3,2.9,2.1,3,3,0,1,0,0,1,0,2,2,0,0,3,3,2,2.5


df2. Conditions_of_people_affected.csv

In [17]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Conditions_of_people_affected.csv
Current file index is: 2


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,# of people in none/minimal conditions - Level 1,# of people in stressed conditions - level 2,# of people in moderate conditions - level 3,# of people severe conditions - level 4,# of people extreme conditions - level 5,People in Need,% of people in none/minimal conditions - Level 1,% of people in stressed conditions - level 2,% of people in moderate conditions - level 3,% of people severe conditions - level 4,% of people extreme conditions - level 5,Concentration of conditions,Conditions of people affected,NaN
2,NaN,NaN,NaN,NaN,MAX,NaN,NaN,NaN,NaN,NaN,7,0.05,0.05,0.05,0.05,0.05,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,MIN,NaN,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CRISIS,TYPE OF CRISIS,CRISIS ID,COUNTRY,Iso3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Conditions_of_people_affected.csv"):
    if df.iloc[1, 5] == "# of people in none/minimal conditions - Level 1" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-39) down into row 4 (columns 5-39)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    # Delete empty rows
    df = clean_dataframe(df, crisis_id_col = 2)
    df2 = df
else: print("Wrong file")

Dropping NaN rows: [0, 1, 2, 3, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145]
Dropping NaN columns: [18]


In [19]:
# Display the updated DataFrame
df2

,Crisis,Type Of Crisis,Crisis Id,Country,Iso3,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected
1,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,0,19.642,12.1,5.8,0.5,5,0,0.5163240628778718,0.3180695021292256,0.15246306713632302,0.01314336785657957,4,4.5
2,Nagorno-Karabakh Conflict in Armenia,Conflict,ARM002,Armenia,Arm,2.874,0.018,0.066,0,0,1.4,0.9503968253968254,0.005952380952380952,0.021825396825396824,0,0,1,1.2
3,Nagorno-Karabakh conflict in Azerbaijan,Conflict,AZE002,Azerbaijan,Aze,x,x,x,x,x,x,x,x,x,x,x,x,x
4,Complex in Burundi,Complex crisis,BDI001,Burundi,Bdi,0,10.2,1.272,1.032,0.096,4,0,0.8095238095238095,0.10095238095238095,0.08190476190476191,0.007619047619047619,4,4
5,Conflict in Burkina Faso,Conflict,BFA002,Burkina Faso,Bfa,0.081,2,0.983,0.175,0.284,3.6,0.022991768379222254,0.5676979846721544,0.27902355946636387,0.04967357365881351,0.08061311382344592,5,4.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,Complex crisis in Venezuela,Complex crisis,VEN001,Venezuela,Ven,0,12.61,14.803,0,0,5,0,0.46001751057930834,0.5400189697942507,0,0,3,4
126,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,3.6,6.4,8.4,8.9,3.4,5,0.11726384364820847,0.20846905537459284,0.2736156351791531,0.2899022801302932,0.11074918566775244,5,5
127,Mixed migration flows in Yemen,International displacement,YEM002,Yemen,Yem,3.6,6.4,8.4,8.9,3.4,5,0.11726384364820847,0.20846905537459284,0.2736156351791531,0.2899022801302932,0.11074918566775244,5,5
128,Drought in Zambia,Drought,ZMB002,Zambia,Zmb,6.25,4.438,1.175,0,0,3.5,0.5268481834274635,0.3741043580881733,0.09904745848436315,0,0,3,3.25


In [20]:
#####-MERGE-#####

merged_df = merge_df(df1, df2, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis                        Type  \
0              Complex crisis in Afghanistan              Complex crisis   
1       Nagorno-Karabakh Conflict in Armenia                    Conflict   
2    Nagorno-Karabakh conflict in Azerbaijan                    Conflict   
3                         Complex in Burundi              Complex crisis   
4                   Conflict in Burkina Faso                    Conflict   
..                                       ...                         ...   
124              Complex crisis in Venezuela              Complex crisis   
125                        Conflict in Yemen              Complex crisis   
126           Mixed migration flows in Yemen  International displacement   
127                        Drought in Zambia                     Drought   
128               Complex crisis in Zimbabwe         

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected
0,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,3.6,3.4,3.5,3.7,0,3.8,x,3.8,3.0499999999999994,10,5,7.5,4.2,4.2,3.5,3.7,3.9,4.8,5,5,0,3,2,3,4,2,3,5,3,3,3,5,5,5,Complex crisis,0,19.642,12.1,5.8,0.5,5,0,0.5163240628778718,0.3180695021292256,0.15246306713632302,0.01314336785657957,4,4.5
1,Nagorno-Karabakh Conflict in Armenia,Conflict,ARM002,Armenia,Arm,2.9,1.5,2.2,0.2,0.2,1.7,0.6,1.15,1.1833333333333333,6,0,3,2.9,2.6,2,2.4,2.5,2.2,2,2,0,0,0,0,0,0,0,0,0,2,0,2,1,1.5,Conflict,2.874,0.018,0.066,0,0,1.4,0.9503968253968254,0.005952380952380952,0.021825396825396824,0,0,1,1.2
2,Nagorno-Karabakh conflict in Azerbaijan,Conflict,AZE002,Azerbaijan,Aze,3.6,3.3,3.45,0.8,0.6,2.1,0.2,1.1500000000000001,1.7666666666666668,10,2.8,6.4,3.5,3.1,3.5,4.5,3.7,4,3,3,2,0,1,0,2,0,0,0,1,3,1,4,2,2.5,Conflict,x,x,x,x,x,x,x,x,x,x,x,x,x
3,Complex in Burundi,Complex crisis,BDI001,Burundi,Bdi,2.1,3.2,2.6500000000000004,1.3,0,3.5,1.7,2.6,1.9666666666666668,6,3.4,4.7,4.1,3.9,3.8,4.4,4.1,3.6,5,5,0,2,0,0,2,0,0,0,0,0,3,3,2,3.5,Complex crisis,0,10.2,1.272,1.032,0.096,4,0,0.8095238095238095,0.10095238095238095,0.08190476190476191,0.007619047619047619,4,4
4,Conflict in Burkina Faso,Conflict,BFA002,Burkina Faso,Bfa,1.1,1.9,1.5,2.8,0,4.1,1.3,2.6999999999999997,1.8666666666666665,10,4,7,3,2.9,2.9,2.2,2.8,3.9,4,4,0,2,2,0,2,0,3,3,2,0,3,4,3,3.5,Conflict,0.081,2,0.983,0.175,0.284,3.6,0.022991768379222254,0.5676979846721544,0.27902355946636387,0.04967357365881351,0.08061311382344592,5,4.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,Complex crisis in Venezuela,Complex crisis,VEN001,Venezuela,Ven,2.5,3.5,3,1.3,1.2,3.1,2.5,2.8,2.35,6,5,5.5,4.2,4.8,4.1,4.2,4.3,4.1,2,2,2,3,2,2,4,3,3,5,3,0,3,4,4,3,Complex crisis,0,12.61,14.803,0,0,5,0,0.46001751057930834,0.5400189697942507,0,0,3,4
125,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,4.3,4.3,4.3,3.1,0,5,1.5,3.25,3.033333333333333,10,3.6,6.8,4.3,4.3,4.4,4.5,4.4,4.7,5,5,2,3,3,2,5,2,3,5,2,1,3,4,5,5,Complex crisis,3.6,6.4,8.4,8.9,3.4,5,0.11726384364820847,0.20846905537459284,0.27

df3. Core_Indicators.csv

In [21]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df


Opening file: october-2021-Core_Indicators.csv
Current file index is: 3


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252
0,Crisis,Type of crisis,Crisis ID,Country,ISO3 code,Total population,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Landmass affected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People exposed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People affected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People displaced,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Injuries reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Illness cases reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fatalities reported,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Buildings damaged,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Buildings destroyed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Buildings in the affected area,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Economic Losses,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People in Need,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Minimal humanitarian conditions - Level 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Stressed humanitarian conditions - Level 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Moderate humanitarian conditions - Level 3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Severe humanitarian conditions - Level 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Extreme humanitarian conditions - Level 5,NaN,NaN,NaN,NaN,NaN,NaN,Fatalities in all crises,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Crisis affected groups,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People facing limited access constraints,NaN,NaN,NaN,NaN,NaN,NaN,NaN,People facing restricted access constraints,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Impediments to entry into country (bureaucrati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Restriction of movement (impediments to freedo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Interference into implementation of humanitari...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Violence against personnel, facilities and assets",NaN,NaN,NaN,NaN,NaN,NaN,NaN,Denial of existence of humanitarian needs or e...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Restriction and obstruction of access to servi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ongoing insecurity/hostilities affecting human...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Presence of mines and improvised explosive dev...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Physical constraints in the environment (obsta...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,NaN,0,NaN,0,Helper 0,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 1,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 2,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 3,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 4,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 5,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 6,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 7,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 8,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 9,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 10,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 11,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 12,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 13,Figure,Source,Reliability,Reliability Score,Justification,Link,Date,Helper 14,Figure,Source,Reliability,Rel

In [22]:
def find_column_name(df, row, col):
    # Start one row up from the "Helper" cell
    current_row = row - 1
    current_col = col  # Starting column

    # Iterate to the right until a non-empty cell is found
    while current_col < df.shape[1]:  # Ensure we don't go beyond the last column
        if pd.notna(df.iloc[current_row, current_col]):
            return df.iloc[current_row, current_col]
        current_col += 1  # Move to the next column

    # If no column name is found, return a placeholder or None
    return None

# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Core_Indicators.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]) and re.match(r"^Helper.*", str(df.loc[1, col])):
        # Try to find the column name by searching upwards
            col_name = find_column_name(df, 1, col)
        df.loc[0, col] =f"{col_name} [{str(df.loc[1, col])}]"
    df = clean_dataframe(df, crisis_id_col = 3)
    
    df3 = df
else: print("Wrong file")

Dropping NaN rows: [1, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187]
Dropping NaN columns: []


In [23]:
df3

,Crisis,Type Of Crisis,Crisis Id,Country,Iso3,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildings Destroyed [Source],Buildings Destroyed [Reliability],Buildings Destroyed [Reliability Score],Buildings Destroyed [Justification],Buildings Destroyed [Link],Buildings Destroyed [Date],Buildings In The Affected Area [Helper 10],Buildings In The Affected Area [Figure],Buildings In The Affected Area [Source],Buildings In The Affected Area [Reliability],Buildings In The Affected Area [Reliability Score],Buildings In The Affected Area [Justification],Buildings In The Affected Area [Link],Buildings In The Affected Area [Date],Economic Losses [Helper 11],Economic Losses [Figure],Economic Losses [Source],Economic Losses [Reliability],Economic Losses [Reliability Score],Economic Losses [Justification],Economic Losses [Link],Economic Losses [Date],People In Need [Helper 12],People In Need [Figure],People In Need [Source],People In Need [Reliability],People In Need [Reliability Score],People In Need [Justification],People In Need [Link],People In Need [Date],Minimal Humanitarian Conditions - Level 1 [Helper 13],Minimal Humanitarian Conditions - Level 1 [Figure],Minimal Humanitarian Conditions - Level 1 [Source],Minimal Humanitarian Conditions - Level 1 [Reliability],Minimal Humanitarian Conditions - Level 1 [Reliability Score],Minimal Humanitarian Conditions - Level 1 [Justification],Minimal Humanitarian Conditions - Level 1 [Link],Minimal Humanitarian Conditions - Level 1 [Date],Stressed Humanitarian Conditions - Level 2 [Helper 14],Stressed Humanitarian Conditions - Level 2 [Figure],Stressed Humanitarian Conditions - Level 2 [Source],Stressed Humanitarian Conditions - Level 2 [Reliability],Stressed Humanitarian Conditions - Level 2 [Reliability Score],Stressed Humanitarian Conditions - Level 2 [Justification],Stressed Humanitarian 

In [24]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df3, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis                        Type  \
0              Complex crisis in Afghanistan              Complex crisis   
1       Nagorno-Karabakh Conflict in Armenia                    Conflict   
2    Nagorno-Karabakh conflict in Azerbaijan                    Conflict   
3                         Complex in Burundi              Complex crisis   
4                   Conflict in Burkina Faso                    Conflict   
..                                       ...                         ...   
124              Complex crisis in Venezuela              Complex crisis   
125                        Conflict in Yemen              Complex crisis   
126           Mixed migration flows in Yemen  International displacement   
127                        Drought in Zambia                     Drought   
128               Complex crisis in Zimbabwe         

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df4. Country_Indicator_Data.csv

In [25]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Country_Indicator_Data.csv
Current file index is: 4


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115
0,Country,ISO3,Ethnic Fractionalisation Index,Empowerment Rights Index,size of excluded ethnic groups,BTI - Democracy Status,Gender Inequality Index,Income Gini coefficient,Conflict Intensity (HIIK),People killed in all crises,Rule of Law (WGI),Rule of Law (BTI),Freedom in the World Index,CPI,Total Population,Land area (sq. km),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Year,NaN,2017,2011,2004 - 2006,2020,2018,2006 - 2018,2019,2020,2019,2020,2020,2019,2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Unit of Measurement,NaN,Index,Index,%,Index,Index,Index,Index,Number,Index,Index,Index,Index,Number,sq. Km,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,0.7497860034,4,0,3.2833333333,0.5747417061,x,10,35638,-1.7135269642,3,27,16,38042000,652860,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,0.7626000092,4,0.62,4.65,0.5779989038,51.3,6,x,-1.0543431044,3.75,32,26,x,1246700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

In [26]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Country_Indicator_Data.csv"):
    cols_to_drop = []
    for col in range(2, len(df.columns)):
        if pd.isna(df.iloc[0, col]):
            cols_to_drop.append(df.columns[col])
            df[col] = df[col].astype(object)  # Convert the entire column to object type
        df.iloc[0, col] =f"{df.iloc[0, col]} [{str(df.iloc[1, col])}] [{df.iloc[2, col]}]"

    print(f"Columns to drop: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

    df = clean_dataframe(df, crisis_id_col = 1)
    df4 = df
else: print("Wrong file")

Columns to drop: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]
Dropping NaN rows: [1, 2, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294]
Dropping NaN columns: []


In [27]:
df4

,Country,Iso3,Ethnic Fractionalisation Index [2017] [Index],Empowerment Rights Index [2011] [Index],Size Of Excluded Ethnic Groups [2004 - 2006] [%],Bti - Democracy Status [2020] [Index],Gender Inequality Index [2018] [Index],Income Gini Coefficient [2006 - 2018] [Index],Conflict Intensity (Hiik) [2019] [Index],People Killed In All Crises [2020] [Number],Rule Of Law (Wgi) [2019] [Index],Rule Of Law (Bti) [2020] [Index],Freedom In The World Index [2020] [Index],Cpi [2019] [Index],Total Population [2020] [Number],Land Area (Sq. Km) [Nan] [Sq. Km]
1,Afghanistan,Afg,0.7497860034,4,0,3.2833333333,0.5747417061,x,10,35638,-1.7135269642,3,27,16,38042000,652860
2,Angola,Ago,0.7626000092,4,0.62,4.65,0.5779989038,51.3,6,x,-1.0543431044,3.75,32,26,x,1246700
3,Albania,Alb,0.320800012,9,0.04,7.15,0.2340778537,33.2,6,x,-0.4111793935,5.5,67,35,x,27400
4,United Arab Emirates,Are,0.9856000006,1,0,3.9,0.1130989843,26,0,x,0.8402188420000001,4.25,17,71,x,83600
5,Argentina,Arg,0.05887494450000001,12,0.01,8.15,0.3537609678,42.9,0,x,-0.4307256341,7.5,85,45,x,2736690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,Samoa,Wsm,0,11,0,x,0.3644043293,38.7,0,x,1.0770641565,x,81,x,x,2830
189,Yemen,Yem,0.6249999927000001,2,0,1.5,0.8341737425,36.7,10,333,-1.7733464241,1.25,11,15,30800000,527970
190,South Africa,Zaf,0.8757249559,11,0,7.45,0.4215473129,63,6,x,-0.0764076784,7.25,79,44,x,1213090
191,Zambia,Zmb,0.7025999877,8,0,5.75,0.5403252994,57.1,0,12,-0.4620692729999999,4.25,54,34,18400000,743390


In [28]:
#####-MERGE-#####
merged_df = merge_df(merged_df, df4, on_col="Country")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                               Crisis                        Type Crisis Id  \
0       Complex crisis in Afghanistan              Complex crisis    AFG001   
1                                 NaN                         NaN       NaN   
2    Mixed migration flows in Algeria  International displacement    DZA002   
3         Sahrawi refugees in Algeria  International displacement    DZA003   
4          Multiple crises in Algeria     Multiple crises country    DZA004   
..                                ...                         ...       ...   
248                               NaN                         NaN       NaN   
249                 Conflict in Yemen              Complex crisis    YEM001   
250    Mixed migration flows in Yemen  International displacement    YEM002   
251                 Drought in Zambia                     Drought    ZMB002   
252        Complex c

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df5. Crisis_Indicator_Data.csv

In [29]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Crisis_Indicator_Data.csv
Current file index is: 5


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
0,Crisis,TYPE OF CRISIS,CRISIS ID,COUNTRY,ISO3,Landmass affected by disaster,People living in the affected area,Total # of people affected by the crisis,Total # of crisis related displaced people,Total # of crisis related Injuries,Total # of illness cases reported,Total # of crisis related fatalities,Building partially damaged,Building totally damaged,Total Building in the affected area,Econimic losses,# of people affected facing minimal humanitari...,# of people affected facing stressed humanitar...,# of people affected facing moderate humanitar...,# of people affected facing severe humanitaria...,# of people affected facing extreme humanitari...,# of groups affected by crisis,people in need facing limited access constraints,people in need facing restricted access constr...,Impediments to entry into country (bureaucrati...,Restriction of movement (impediments to freedo...,Interference into implementation of humanitari...,"Violence against personnel, facilities and assets",Denial of existence of humanitarian needs or e...,Restriction and obstruction of access to servi...,Ongoing insecurity/hostilities affecting human...,Presence of mines and improvised explosive dev...,Physical constraints in the environment (obsta...
1,0,NaN,NaN,NaN,NaN,sq. Km,Number,Number,Number,Number,NaN,Number,Number,Number,Number,million USD,Number,Number,Number,Number,Number,Number,Number,Number,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,AFG,652867,38042000,38042000,8493000,x,x,35638,x,x,NaN,13114,0,19642000,12100000,5800000,500000,5,x,x,0.0,3.0,2.0,3.0,2.0,3.0,3.0,3.0,3.0
3,Nagorno-Karabakh Conflict in Armenia,Conflict,ARM002,Armenia,ARM,28203,3024000,84000,66000,x,x,1,x,x,NaN,x,2874000,18000,66000,0,0,2,x,x,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
4,Nagorno-Karabakh conflict in Azerbaijan,Conflict,AZE002,Azerbaijan,AZE,31560,5874000,5874000,91000,x,x,91,x,x,NaN,x,x,x,x,x,x,3,x,x,2.0,0.0,1.0,0.0,0.0,0.0,1.0,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,Complex crisis in Venezuela,Complex crisis,VEN001,Venezuela,VEN,882050,27412000,27412000,5443000,x,x,8253,x,x,NaN,223750,0,12610000,14803000,0,0,2,x,x,2.0,3.0,2.0,2.0,3.0,3.0,3.0,0.0,3.0
127,Conflict in Yemen,Complex crisis,YEM001,Yemen,YEM,527970,30700000,27100000,4000000,743,19116,333,x,x,NaN,x,3600000,6400000,8400000,8900000,3400000,5,x,x,2.0,3.0,3.0,2.0,2.0,3.0,2.0,1.0,3.0
128,Mixed migration flows in Yemen,International displacement,YEM002,Yemen,YEM,527970,30700000,27100000,4000000,743,19116,333,x,x,NaN,x,3600000,6400000,8400000,8900000,3400000,2,x,x,2.0,3.0,3.0,2.0,2.0,2.0,2.0,1.0,3.0
129,Drought in Zambia,Drought,ZMB002,Zambia,ZMB,752618,11863000,5613000,0,0,0,0,0,0,NaN,x,6250000,4438000,1175000,0,0,3,x,x,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,3.0


In [30]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Crisis_Indicator_Data.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]):
            df.loc[0, col] =f"{df.loc[0, col]} [{df.loc[1, col]}]" 
    df = clean_dataframe(df, crisis_id_col = 2)
    df5 = df
else: print("Wrong file")

Dropping NaN rows: [1]
Dropping NaN columns: []


In [31]:
df5

,Crisis,Type Of Crisis,Crisis Id,Country,Iso3,Landmass Affected By Disaster [Sq. Km],People Living In The Affected Area [Number],Total # Of People Affected By The Crisis [Number],Total # Of Crisis Related Displaced People [Number],Total # Of Crisis Related Injuries [Number],Total # Of Illness Cases Reported,Total # Of Crisis Related Fatalities [Number],Building Partially Damaged [Number],Building Totally Damaged [Number],Total Building In The Affected Area [Number],Econimic Losses [Million Usd],# Of People Affected Facing Minimal Humanitarian Needs (Level 1) [Number],# Of People Affected Facing Stressed Humanitarian Needs (Level 2) [Number],# Of People Affected Facing Moderate Humanitarian Conditions And Needs (Level 3) [Number],# Of People Affected Facing Severe Humanitarian Conditions And Needs (Level 4) [Number],# Of People Affected Facing Extreme Humanitarian Conditions And Needs (Level 5) [Number],# Of Groups Affected By Crisis [Number],People In Need Facing Limited Access Constraints [Number],People In Need Facing Restricted Access Constraints [Number],Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)"
1,Complex Crisis In Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,652867,38042000,38042000,8493000,x,x,35638,x,x,NaN,13114,0,19642000,12100000,5800000,500000,5,x,x,0.0,3.0,2.0,3.0,2.0,3.0,3.0,3.0,3.0
2,Nagorno-Karabakh Conflict In Armenia,Conflict,ARM002,Armenia,Arm,28203,3024000,84000,66000,x,x,1,x,x,NaN,x,2874000,18000,66000,0,0,2,x,x,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
3,Nagorno-Karabakh Conflict In Azerbaijan,Conflict,AZE002,Azerbaijan,Aze,31560,5874000,5874000,91000,x,x,91,x,x,NaN,x,x,x,x,x,x,3,x,x,2.0,0.0,1.0,0.0,0.0,0.0,1.0,3.0,1.0
4,Complex In Burundi,Complex crisis,BDI001,Burundi,Bdi,25680,12600000,12600000,416000,x,70187,262,x,x,NaN,444,0,10200000,1272000,1032000,96000,5,x,x,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
5,Conflict In Burkina Faso,Conflict,BFA002,Burkina Faso,Bfa,166445,3523000,3442000,1241000,x,x,3145,x,x,NaN,x,81000,2000000,983000,175000,284000,4,x,x,0.0,2.0,2.0,0.0,0.0,3.0,2.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,Complex Crisis In Venezuela,Complex crisis,VEN001,Venezuela,Ven,882050,27412000,27412000,5443000,x,x,8253,x,x,NaN,223750,0,12610000,14803000,0,0,2,x,x,2.0,3.0,2.0,2.0,3.0,3.0,3.0,0.0,3.0
126,Conflict In Yemen,Complex crisis,YEM001,Yemen,Yem,527970,30700000,27100000,4000000,743,19116,333,x,x,NaN,x,3600000,6400000,8400000,8900000,3400000,5,x,x,2.0,3.0,3.0,2.0,2.0,3.0,2.0,1.0,3.0
127,Mixed Migration Flows In Yemen,International displacement,YEM002,Yemen,Yem,527970,30700000,27100000,4000000,743,19116,333,x,x,NaN,x,3600000,6400000,8400000,8900000,3400000,2,x,x,2.0,3.0,3.0,2.0,2.0,2.0,2.0,1.0,3.0
128,Drought In Zambia,Drought,ZMB002,Zambia,Zmb,752618,11863000,5613000,0,0,0,0,0,0,NaN,x,6250000,4438000,1175000,0,0,3,x,x,0.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,3.0


In [32]:
merged_df = merge_df(merged_df, df5, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                                      NaN             NaN       NaN   
250                                      NaN             NaN       NaN   
251                                      NaN             NaN       NaN   
252                                      NaN             NaN       NaN   



,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df6. Crisis_info.csv

In [33]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Crisis_info.csv
Current file index is: 6


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Crisis ID,GlideNumber,SeverityGlideNumber,Country,Crisis,ISO3 code,Region,Type of crisis,First country,Website name,Crisis description,Related crisis,Starting date,Duration (days)
1,AFG001,NaN,NaN,Afghanistan,Complex crisis in Afghanistan,AFG,Asia,Complex crisis,Afghanistan,Complex crisis,https://www.acaps.org/country/Afghanistan/cris...,NaN,Before 2019,More than 2 years
2,ARM002,NaN,NaN,Armenia,Nagorno-Karabakh Conflict in Armenia,ARM,Middle east,Conflict,Armenia,Nagorno-Karabakh Conflict in Armenia,https://www.acaps.org/country/Armenia/crisis/N...,NaN,04/11/2020,361
3,AZE002,NaN,NaN,Azerbaijan,Nagorno-Karabakh conflict in Azerbaijan,AZE,Middle east,Conflict,Azerbaijan,Nagorno-Karabakh conflict in Azerbaijan,https://www.acaps.org/country/Azerbaijan/crisi...,NaN,04/11/2020,361
4,BDI001,NaN,NaN,Burundi,Complex in Burundi,BDI,Africa,Complex crisis,Burundi,Complex crisis,https://www.acaps.org/country/Burundi/crisis/C...,NaN,Before 2019,More than 2 years
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,VEN001,NaN,NaN,Venezuela,Complex crisis in Venezuela,VEN,Americas,Complex crisis,Venezuela,Complex crisis,https://www.acaps.org/country/Venezuela/crisis...,NaN,Before 2019,More than 2 years
126,YEM001,NaN,NaN,Yemen,Conflict in Yemen,YEM,Middle east,Complex crisis,Yemen,Complex crisis,https://www.acaps.org/country/Yemen/crisis/Com...,NaN,Before 2019,More than 2 years
127,YEM002,NaN,NaN,Yemen,Mixed migration flows in Yemen,YEM,Middle east,International displacement,Yemen,Mixed Migration,https://www.acaps.org/country/Yemen/crisis/Mix...,NaN,01/02/2019,1003
128,ZMB002,NaN,NaN,Zambia,Drought in Zambia,ZMB,Africa,Drought,Zambia,Drought,https://www.acaps.org/country/Zambia/crisis/Dr...,NaN,Before 2019,More than 2 years


In [34]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Crisis_info.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df6 = df
else: print("Wrong file")

Dropping NaN rows: []
Dropping NaN columns: []


In [35]:
df6

,Crisis Id,Glidenumber,Severityglidenumber,Country,Crisis,Iso3,Region,Type Of Crisis,First Country,Website Name,Crisis Description,Related Crisis,Starting Date,Duration (Days)
1,AFG001,NaN,NaN,Afghanistan,Complex Crisis In Afghanistan,Afg,Asia,Complex crisis,Afghanistan,Complex crisis,https://www.acaps.org/country/Afghanistan/cris...,NaN,Before 2019,More than 2 years
2,ARM002,NaN,NaN,Armenia,Nagorno-Karabakh Conflict In Armenia,Arm,Middle east,Conflict,Armenia,Nagorno-Karabakh Conflict in Armenia,https://www.acaps.org/country/Armenia/crisis/N...,NaN,04/11/2020,361
3,AZE002,NaN,NaN,Azerbaijan,Nagorno-Karabakh Conflict In Azerbaijan,Aze,Middle east,Conflict,Azerbaijan,Nagorno-Karabakh conflict in Azerbaijan,https://www.acaps.org/country/Azerbaijan/crisi...,NaN,04/11/2020,361
4,BDI001,NaN,NaN,Burundi,Complex In Burundi,Bdi,Africa,Complex crisis,Burundi,Complex crisis,https://www.acaps.org/country/Burundi/crisis/C...,NaN,Before 2019,More than 2 years
5,BFA002,NaN,NaN,Burkina Faso,Conflict In Burkina Faso,Bfa,Africa,Conflict,Burkina Faso,Conflict,https://www.acaps.org/country/Burkina Faso/cri...,NaN,Before 2019,More than 2 years
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,VEN001,NaN,NaN,Venezuela,Complex Crisis In Venezuela,Ven,Americas,Complex crisis,Venezuela,Complex crisis,https://www.acaps.org/country/Venezuela/crisis...,NaN,Before 2019,More than 2 years
126,YEM001,NaN,NaN,Yemen,Conflict In Yemen,Yem,Middle east,Complex crisis,Yemen,Complex crisis,https://www.acaps.org/country/Yemen/crisis/Com...,NaN,Before 2019,More than 2 years
127,YEM002,NaN,NaN,Yemen,Mixed Migration Flows In Yemen,Yem,Middle east,International displacement,Yemen,Mixed Migration,https://www.acaps.org/country/Yemen/crisis/Mix...,NaN,01/02/2019,1003
128,ZMB002,NaN,NaN,Zambia,Drought In Zambia,Zmb,Africa,Drought,Zambia,Drought,https://www.acaps.org/country/Zambia/crisis/Dr...,NaN,Before 2019,More than 2 years


In [36]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df6, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                                      NaN             NaN       NaN   
250                                      NaN             NaN       NaN   
251                                      NaN             NaN       NaN   
252                                      NaN             NaN       NaN   



,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df7. Data_Reliability.csv

In [37]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Data_Reliability.csv
Current file index is: 7


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CrisisID,Data Reliability,Information gaps (# indicator missing),Impact of the crisis,Area affected,People in the affected area,Geographical,People affected,People displaced,Crisis related fatalities,People affected by categories,Human,Conditions of people affected,# of people facing minimal humanitarian needs ...,# of people facing stressed humanitarian condi...,# of people facing moderate humanitarian condi...,# of people facing severe humanitarian conditi...,# of people facing extreme humanitarian condit...,Complexity of the crisis,Society and safety,Social cohesion,Conflict Intensity,Total killed in all crisis,Safety and security,Rule of Law,Operating environment,Diversity of groups affected,Humanitarian Access,Impediments to entry into country (bureaucrati...,Restriction of movement (impediments to freedo...,Interference into implementation of humanitari...,"Violence against personnel, facilities and assets",Access of Humanitarian Actors to Affected Popu...,Denial of existence of humanitarian needs or e...,Restriction and obstruction of access to servi...,Access of People in need to Aid,Ongoing insecurity/hostilities affecting human...,Presence of mines and improvised explosive dev...,Physical constraints in the environment (obsta...,Physical and Security Constraints
2,AFG001,2,0,2.2,1,2,1.5,2,3,3,3,2.5,2,2,2,2,2,2,1.7,1.3333333333333333,1,1,3,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
3,ARM002,1.8,0,1.8,1,2,1.5,2,2,2,2,2,2,2,2,2,2,2,1.6,1.1666666666666667,1,1,2,1.5,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
4,AZE002,x,2,2.6,1,3,2.1,3,2,3,2.5,2.8,x,x,x,x,x,x,1.7,1.3333333333333333,1,1,3,2,1,2,x,2,2,2,2,2,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!
198,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!
199,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!
200,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!


In [38]:
# Case-Specific
current_file = file_list[current_file_index]
if current_file.endswith("Data_Reliability.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df7 = df
else: print("Wrong file")

Dropping NaN rows: [0]
Dropping NaN columns: []


In [39]:
df7['Crisis Id'] = [id.upper() for id in df7['Crisis Id']]
df7

,Crisis Id,Data Reliability,Information Gaps (# Indicator Missing),Impact Of The Crisis,Area Affected,People In The Affected Area,Geographical,People Affected,People Displaced,Crisis Related Fatalities,People Affected By Categories,Human,Conditions Of People Affected,# Of People Facing Minimal Humanitarian Needs (Level 1),# Of People Facing Stressed Humanitarian Conditions And Needs (Level 2),# Of People Facing Moderate Humanitarian Conditions And Needs (Level 3),# Of People Facing Severe Humanitarian Conditions And Needs (Level 4),# Of People Facing Extreme Humanitarian Conditions And Needs (Level 5),Complexity Of The Crisis,Society And Safety,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Rule Of Law,Operating Environment,Diversity Of Groups Affected,Humanitarian Access,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints
1,AFG001,2,0,2.2,1,2,1.5,2,3,3,3,2.5,2,2,2,2,2,2,1.7,1.3333333333333333,1,1,3,2,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
2,ARM002,1.8,0,1.8,1,2,1.5,2,2,2,2,2,2,2,2,2,2,2,1.6,1.1666666666666667,1,1,2,1.5,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
3,AZE002,x,2,2.6,1,3,2.1,3,2,3,2.5,2.8,x,x,x,x,x,x,1.7,1.3333333333333333,1,1,3,2,1,2,x,2,2,2,2,2,2,2,2,2,2,2,2,2
4,BDI001,1.9,0,2,1,2,1.5,2,3,2,2.5,2.3,2,2,2,2,2,2,1.6,1.1666666666666667,1,1,2,1.5,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
5,BFA002,1.2,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1.6,1.1666666666666667,1,1,2,1.5,1,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!
197,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!
198,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!
199,0,0,0,0,0,0,0,0,0,0,0,0,#DIV/0!,0,0,0,0,0,0,#DIV/0!,1,1,0,#DIV/0!,1,#DIV/0!,0,#DIV/0!,0,0,0,0,#DIV/0!,0,0,#DIV/0!,0,0,0,#DIV/0!


In [40]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df7, on_col="Crisis Id")
merged_df

Number of rows deleted: 71
Indices of deleted duplicate rows: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70]
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                     

,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df8. Impact_of_the_crisis.csv

In [41]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Impact_of_the_crisis.csv
Current file index is: 8


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,Area affected - absolute,% of total area affected,Area affected - relative,Area affected,People living in the affected area - absolute,% of total population living in the affected area,People living in the affected area - relative,People in the affected area,Geographical,People affected - absolute,% of people affected on the total population e...,People affected - relative,People affected,People displaced - absolute,% of total population displaced on the,People displaced - relative,People displaced,Fatalities - absolute,% of fatalities on the total population affected,Fatalities (relative),Crisis related fatalities,People affected by categories,Human,NaN
2,NaN,NaN,NaN,NaN,MAX,6,NaN,1,NaN,7.5,NaN,1,NaN,NaN,7.5,NaN,1,NaN,6.5,NaN,0.15,NaN,3.5,NaN,0.0001,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,MIN,3,NaN,0.02,NaN,6,NaN,0.01,NaN,NaN,4.5,NaN,0.01,NaN,3,NaN,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN
4,CRISIS,TYPE OF CRISIS,CRISIS ID,COUNTRY,Iso3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Impact_of_the_crisis.csv"):
    if df.iloc[1, 5] == "Area affected - absolute" and pd.isna(df.iloc[4, 5]):
        # Copy the content from row 1 (columns 5-...) down into row 4 (columns 5-...)
        df.iloc[4, 5:df.shape[1]] = df.iloc[1, 5:df.shape[1]]
    df = clean_dataframe(df, crisis_id_col = 0)
    df8 = df
else: print("Wrong file")

Dropping NaN rows: [0, 1, 2, 3, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146]
Dropping NaN columns: [28]


In [43]:
df8

,Crisis,Type Of Crisis,Crisis Id,Country,Iso3,Area Affected - Absolute,% Of Total Area Affected,Area Affected - Relative,Area Affected,People Living In The Affected Area - Absolute,% Of Total Population Living In The Affected Area,People Living In The Affected Area - Relative,People In The Affected Area,Geographical,People Affected - Absolute,% Of People Affected On The Total Population Exposed,People Affected - Relative,People Affected,People Displaced - Absolute,% Of Total Population Displaced On The,People Displaced - Relative,People Displaced,Fatalities - Absolute,% Of Fatalities On The Total Population Affected,Fatalities (Relative),Crisis Related Fatalities,People Affected By Categories,Human
1,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,Afg,4.7,1.0000107220537329,5,4.85,5,1,5,5,4.9,5,1,5,5,5,0.22325324641186056,5,5,5,0.0009368066873455654,5,5,5,5
2,Nagorno-Karabakh Conflict in Armenia,Conflict,ARM002,Armenia,Arm,2.4,0.9906217070600633,5,3.7,1.6,1,5,3.3,3.5,0.7,0.027777777777777776,0.1,0.39999999999999997,2.6,0.7857142857142857,5,3.8,0,1.1904761904761905e-05,0.6,0.3,2.4,1.5
3,Nagorno-Karabakh conflict in Azerbaijan,Conflict,AZE002,Azerbaijan,Aze,2.5,0.38179112783228286,1.8,2.15,2.6,0.5874587458745875,2.9,2.75,2.5,3.8,1,5,4.4,2.8,0.015491998638066053,0.5,1.7,2.8,1.5491998638066055e-05,0.8,1.8,1.8,3.4
4,Complex in Burundi,Complex crisis,BDI001,Burundi,Bdi,2.3,1,5,3.65,3.7,1,5,4.35,4,4.3,1,5,4.65,3.7,0.03301587301587302,1.1,2.4,3.5,2.0793650793650793e-05,1,2.3,2.4,3.8
5,Conflict in Burkina Faso,Conflict,BFA002,Burkina Faso,Bfa,3.7,0.6083516081871345,3,3.35,1.8,0.16669032410693163,0.8,1.3,2.5,3.4,0.9770082316207778,4.9,4.15,4.4,0.36054619407321326,5,4.7,5,0.0009137129575828007,5,5,4.9,4.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,Complex crisis in Venezuela,Complex crisis,VEN001,Venezuela,Ven,4.9,1,5,4.95,4.8,1,5,4.9,4.9,4.9,1,5,4.95,5,0.1985626732817744,5,5,5,0.00030107252298263533,5,5,5,5
126,Conflict in Yemen,Complex crisis,YEM001,Yemen,Yem,4.5,1,5,4.75,5,0.9967532467532467,5,5,4.9,4.9,0.8827361563517915,4.4,4.65,5,0.14760147601476015,4.9,5,3.6,1.2287822878228782e-05,0.6,2.1,4,4.4
127,Mixed migration flows in Yemen,International displacement,YEM002,Yemen,Yem,4.5,1,5,4.75,5,0.9967532467532467,5,5,4.9,4.9,0.8827361563517915,4.4,4.65,5,0.14760147601476015,4.9,5,3.6,1.2287822878228782e-05,0.6,2.1,4,4.4
128,Drought in Zambia,Drought,ZMB002,Zambia,Zmb,4.8,1.0124134034625163,5,4.9,3.6,0.6447282608695653,3.2,3.4000000000000004,4.3,3.7,0.47315181657253647,2.3,3,0,0,0,0,0,0,0,0,0,1.7


In [44]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df8, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                                      NaN             NaN       NaN   
250                                      NaN             NaN       NaN   
251                                      NaN             NaN       NaN   
252                                      NaN             NaN       NaN   



,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

Imputed_and_missing_data_hidden.csv -- No data

In [45]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Opening file: october-2021-Imputed_and_missing_data_hidden.csv
Current file index is: 9


Indicator_Date_hidden.csv -- No data

In [46]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Opening file: october-2021-Indicator_Date_hidden.csv
Current file index is: 10


Indicator_Date_hidden2.csv -- No data

In [47]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Opening file: october-2021-Indicator_Date_hidden2.csv
Current file index is: 11


Indicator_Metadata.csv -- No data, just description of features

In [48]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")


Opening file: october-2021-Indicator_Metadata.csv
Current file index is: 12


In [49]:
df = clean_dataframe(df, crisis_id_col = 0)
df.head()

Dropping NaN rows: [0]
Dropping NaN columns: []


,Dimension,Category,Component,Indicator Name,Description,Relevance,Validity / Limitation Of Indicator,Data Sources,Citation,Url
1,Inpact of the Crisis,Geographical Impact,Area affected,Area affected by disaster,Total # of square kilometers affected by the c...,The Impact of crisis dimension reflects the im...,NaN,OCHA; GDACS; UNOSAT; ACLED; INSO; HIIK,NaN,NaN
2,Inpact of the Crisis,Geographical Impact,Area affected,Area affected by disaster (relative),% of square kilometers affected by the crisis ...,The Impact of crisis dimension reflects the im...,NaN,OCHA; GDACS; UNOSAT; ACLED; INSO; HIIK,NaN,NaN
3,Inpact of the Crisis,Geographical Impact,People in the affected area,People living in the affected area,Total # of people living in the affected area,The Impact of crisis dimension reflects the im...,NaN,OCHA; GDACS; USGS; UNOSAT; ACLED; INSO; HIIK; ...,NaN,NaN
4,Inpact of the Crisis,Geographical Impact,People in the affected area,People living in the affected area (relative),% of people living in the affected area on the...,The Impact of crisis dimension reflects the im...,NaN,OCHA; GDACS; USGS; UNOSAT; ACLED; INSO; HIIK; ...,NaN,NaN
5,Inpact of the Crisis,Human impact,People affected,People affected,Total # of people affected by the crisis,The Impact of crisis dimension reflects the im...,NaN,Government; OCHA; Clusters,NaN,NaN


df9. all_crises.csv

In [50]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-INFORM_Severity_all_crises.csv
Current file index is: 13


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,INFORM Severity Index - all crises. Release: O...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CRISIS,CRISIS ID,COUNTRY,ISO3,TYPE OF CRISIS,INFORM Severity Index,INFORM Severity category,INFORM Severity category,Trend (last 3 months),Reliability,Impact of the crisis,Geographical,Human,Conditions of people affected,People in need,Concentration of conditions,Complexity of the crisis,Society and safety,Operating environment,Regions,Last updated
2,Weights,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(a-z),NaN,NaN,(a-z),NaN,(1-5),(1-5),(Very Low-Very High),(Decreasing-Stable-Increasing),(Very Low-Very High),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),NaN,NaN
4,Complex crisis in Afghanistan,AFG001,Afghanistan,AFG,Complex crisis,4.7,5,Very High,Stable,High,5,4.9,5,4.5,5,4,4.9,4.8,5,Asia,2021-09-15 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("all_crises.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df9 = df
else: print("Wrong file")

Dropping NaN rows: [0, 2, 3, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131]
Dropping NaN columns: []


In [52]:
df9

,Crisis,Crisis Id,Country,Iso3,Type Of Crisis,Inform Severity Index,Inform Severity Category,Inform Severity Category,Trend (Last 3 Months),Reliability,Impact Of The Crisis,Geographical,Human,Conditions Of People Affected,People In Need,Concentration Of Conditions,Complexity Of The Crisis,Society And Safety,Operating Environment,Regions,Last Updated
1,Complex crisis in Afghanistan,AFG001,Afghanistan,Afg,Complex crisis,4.7,5,Very High,Stable,High,5,4.9,5,4.5,5,4,4.9,4.8,5,Asia,2021-09-15 00:00:00
2,Nagorno-Karabakh Conflict in Armenia,ARM002,Armenia,Arm,Conflict,1.7,2,Low,Stable,High,2.3,3.5,1.5,1.2,1.4,1,1.9,2.2,1.5,Middle east,2021-08-05 00:00:00
3,Nagorno-Karabakh conflict in Azerbaijan,AZE002,Azerbaijan,Aze,Conflict,x,x,x,-,Low,3.1,2.5,3.4,x,x,x,3.3,4,2.5,Middle east,2021-06-02 00:00:00
4,Complex in Burundi,BDI001,Burundi,Bdi,Complex crisis,3.9,4,High,Increasing,Medium,3.9,4,3.8,4,4,4,3.6,3.6,3.5,Africa,2021-06-09 00:00:00
5,Conflict in Burkina Faso,BFA002,Burkina Faso,Bfa,Conflict,4.1,5,Very High,Increasing,High,4.1,2.5,4.6,4.3,3.6,5,3.7,3.9,3.5,Africa,2021-06-30 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,Complex crisis in Venezuela,VEN001,Venezuela,Ven,Complex crisis,4.2,5,Very High,Increasing,Medium,5,4.9,5,4,5,3,3.6,4.1,3,Americas,2021-06-09 00:00:00
113,Conflict in Yemen,YEM001,Yemen,Yem,Complex crisis,4.9,5,Very High,Stable,High,4.6,4.9,4.4,5,5,5,4.9,4.7,5,Middle east,2021-09-29 00:00:00
114,Mixed migration flows in Yemen,YEM002,Yemen,Yem,International displacement,4.6,5,Very High,Increasing,High,4.6,4.9,4.4,5,5,5,4,4.7,3,Middle east,2021-09-29 00:00:00
115,Drought in Zambia,ZMB002,Zambia,Zmb,Drought,2.9,3,Medium,Increasing,Very High,2.8,4.3,1.7,3.25,3.5,3,2.3,2.1,2.5,Africa,2021-09-12 00:00:00


In [53]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df9, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                                      NaN             NaN       NaN   
250                                      NaN             NaN       NaN   
251                                      NaN             NaN       NaN   
252                                      NaN             NaN       NaN   



,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df10. country.csv

In [54]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-INFORM_Severity_country.csv
Current file index is: 14


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
0,INFORM Severity Index - all countries. Release...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CRISIS,CRISIS ID,COUNTRY,ISO3,TYPE OF CRISIS,INFORM Severity Index,INFORM Severity category,INFORM Severity category,Trend (last 3 months),Reliability,Impact of the crisis,Geographical,Human,Conditions of people affected,People in need,Concentration of conditions,Complexity of the crisis,Society and safety,Operating environment,Regions,Last updated
2,Weights,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(a-z),NaN,NaN,(a-z),NaN,(1-5),(1-5),(Very Low-Very High),(Decreasing-Stable-Increasing),(Very Low-Very High),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),NaN,NaN
4,Complex crisis in Afghanistan,AFG001,Afghanistan,AFG,Complex crisis,4.7,5,Very High,Stable,High,5,4.9,5,4.5,5,4,4.9,4.8,5,Asia,2021-09-15 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,Drought in Zambia,ZMB002,Zambia,ZMB,Drought,2.9,3,Medium,Increasing,Very High,2.8,4.3,1.7,3.25,3.5,3,2.3,2.1,2.5,Africa,2021-09-12 00:00:00
75,Complex crisis in Zimbabwe,ZWE001,Zimbabwe,ZWE,Complex crisis,3.5,4,High,Stable,High,3.5,4.6,2.7,3.85,4.7,3,2.9,3.2,2.5,Africa,2021-09-12 00:00:00
76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("country.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df10 = df
else: print("Wrong file")

Dropping NaN rows: [0, 2, 3, 76, 77, 78]
Dropping NaN columns: []


In [56]:
df10

,Crisis,Crisis Id,Country,Iso3,Type Of Crisis,Inform Severity Index,Inform Severity Category,Inform Severity Category,Trend (Last 3 Months),Reliability,Impact Of The Crisis,Geographical,Human,Conditions Of People Affected,People In Need,Concentration Of Conditions,Complexity Of The Crisis,Society And Safety,Operating Environment,Regions,Last Updated
1,Complex crisis in Afghanistan,AFG001,Afghanistan,Afg,Complex crisis,4.7,5,Very High,Stable,High,5,4.9,5,4.5,5,4,4.9,4.8,5,Asia,2021-09-15 00:00:00
2,Nagorno-Karabakh Conflict in Armenia,ARM002,Armenia,Arm,Conflict,1.7,2,Low,Stable,High,2.3,3.5,1.5,1.2,1.4,1,1.9,2.2,1.5,Middle east,2021-08-05 00:00:00
3,Nagorno-Karabakh conflict in Azerbaijan,AZE002,Azerbaijan,Aze,Conflict,x,x,x,-,Low,3.1,2.5,3.4,x,x,x,3.3,4,2.5,Middle east,2021-06-02 00:00:00
4,Complex in Burundi,BDI001,Burundi,Bdi,Complex crisis,3.9,4,High,Increasing,Medium,3.9,4,3.8,4,4,4,3.6,3.6,3.5,Africa,2021-06-09 00:00:00
5,Conflict in Burkina Faso,BFA002,Burkina Faso,Bfa,Conflict,4.1,5,Very High,Increasing,High,4.1,2.5,4.6,4.3,3.6,5,3.7,3.9,3.5,Africa,2021-06-30 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,Conflict in Ukraine,UKR002,Ukraine,Ukr,Conflict,3.6,4,High,Increasing,High,3.2,1.7,3.7,4.1,4.2,4,3,3,3,Europe,2021-05-30 00:00:00
69,Complex crisis in Venezuela,VEN001,Venezuela,Ven,Complex crisis,4.2,5,Very High,Increasing,Medium,5,4.9,5,4,5,3,3.6,4.1,3,Americas,2021-06-09 00:00:00
70,Conflict in Yemen,YEM001,Yemen,Yem,Complex crisis,4.9,5,Very High,Stable,High,4.6,4.9,4.4,5,5,5,4.9,4.7,5,Middle east,2021-09-29 00:00:00
71,Drought in Zambia,ZMB002,Zambia,Zmb,Drought,2.9,3,Medium,Increasing,Very High,2.8,4.3,1.7,3.25,3.5,3,2.3,2.1,2.5,Africa,2021-09-12 00:00:00


In [57]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df10, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                                      NaN             NaN       NaN   
250                                      NaN             NaN       NaN   
251                                      NaN             NaN       NaN   
252                                      NaN             NaN       NaN   



,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

hidden.csv -- Was hidden. No Data to be used

In [58]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-INFORM_Severity_hidden.csv
Current file index is: 15


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CRISIS,Website name,CRISIS ID,COUNTRY,ISO3,TYPE OF CRISIS,INFORM Severity Index,INFORM Severity category,INFORM Severity category,Trend (last 3 months),Reliability,Impact of the crisis,Geographical Impact,Human Impact,Conditions of people affected,People in need,Concentration of conditions,Complexity of the crisis,Society and safety,Operating environment,Regions,Last updated
2,Weights,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,NaN,NaN,NaN,NaN
3,(a-z),NaN,NaN,NaN,(a-z),NaN,(1-5),(1-5),(Very Low-Very High),(Decreasing-Stable-Increasing),(Very Low-Very High),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),(1-5),NaN,NaN
4,Complex crisis in Afghanistan,Complex crisis,AFG001,Afghanistan,AFG,Complex crisis,4.7,5,Very High,Stable,High,5,4.9,5,4.5,5,4,4.9,4.8,5,Asia,2021-09-15 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,Complex crisis in Venezuela,Complex crisis,VEN001,Venezuela,VEN,Complex crisis,4.2,5,Very High,Increasing,Medium,5,4.9,5,4,5,3,3.6,4.1,3,Americas,2021-06-09 00:00:00
129,Conflict in Yemen,Complex crisis,YEM001,Yemen,YEM,Complex crisis,4.9,5,Very High,Stable,High,4.6,4.9,4.4,5,5,5,4.9,4.7,5,Middle east,2021-09-29 00:00:00
130,Mixed migration flows in Yemen,Mixed Migration,YEM002,Yemen,YEM,International displacement,4.6,5,Very High,Increasing,High,4.6,4.9,4.4,5,5,5,4,4.7,3,Middle east,2021-09-29 00:00:00
131,Drought in Zambia,Drought,ZMB002,Zambia,ZMB,Drought,2.9,3,Medium,Increasing,Very High,2.8,4.3,1.7,3.25,3.5,3,2.3,2.1,2.5,Africa,2021-09-12 00:00:00


df11. Lists.csv

In [59]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Lists.csv
Current file index is: 16


,0,1,2,3,4,5,6,7,8,9,10,11
0,Country,Crisis,Crisis ID,ISO3 code,Region,Type of crisis,Website name,Country level,Individual or aggregated,Indicator,Reliability,Type of entry
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,Complex crisis in Afghanistan,AFG001,AFG,Asia,Complex crisis,Complex crisis,Yes,Individual,Total population,x,Humanitarian development
3,Armenia,Nagorno-Karabakh Conflict in Armenia,ARM002,ARM,Middle east,Conflict,Nagorno-Karabakh Conflict in Armenia,Yes,Individual,Landmass affected,Low,Change in the data/methodology
4,Azerbaijan,Nagorno-Karabakh conflict in Azerbaijan,AZE002,AZE,Middle east,Conflict,Nagorno-Karabakh conflict in Azerbaijan,Yes,Individual,People exposed,Medium,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
126,Venezuela,Complex crisis in Venezuela,VEN001,VEN,Americas,Complex crisis,Complex crisis,Yes,Individual,NaN,NaN,NaN
127,Yemen,Conflict in Yemen,YEM001,YEM,Middle east,Complex crisis,Complex crisis,Yes,Individual,NaN,NaN,NaN
128,Yemen,Mixed migration flows in Yemen,YEM002,YEM,Middle east,International displacement,Mixed Migration,No,Individual,NaN,NaN,NaN
129,Zambia,Drought in Zambia,ZMB002,ZMB,Africa,Drought,Drought,Yes,Individual,NaN,NaN,NaN


In [60]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Lists.csv"):
    df = clean_dataframe(df, crisis_id_col = 2)
    df11 = df
else: print("Wrong file")

Dropping NaN rows: [1]
Dropping NaN columns: []


In [61]:
df11

,Country,Crisis,Crisis Id,Iso3,Region,Type Of Crisis,Website Name,Country Level,Individual Or Aggregated,Indicator,Reliability,Type Of Entry
1,Afghanistan,Complex Crisis In Afghanistan,AFG001,Afg,Asia,Complex crisis,Complex crisis,Yes,Individual,Total population,x,Humanitarian development
2,Armenia,Nagorno-Karabakh Conflict In Armenia,ARM002,Arm,Middle east,Conflict,Nagorno-Karabakh Conflict in Armenia,Yes,Individual,Landmass affected,Low,Change in the data/methodology
3,Azerbaijan,Nagorno-Karabakh Conflict In Azerbaijan,AZE002,Aze,Middle east,Conflict,Nagorno-Karabakh conflict in Azerbaijan,Yes,Individual,People exposed,Medium,NaN
4,Burundi,Complex In Burundi,BDI001,Bdi,Africa,Complex crisis,Complex crisis,Yes,Individual,People affected,High,NaN
5,Burkina Faso,Conflict In Burkina Faso,BFA002,Bfa,Africa,Conflict,Conflict,Yes,Individual,People displaced,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
125,Venezuela,Complex Crisis In Venezuela,VEN001,Ven,Americas,Complex crisis,Complex crisis,Yes,Individual,NaN,NaN,NaN
126,Yemen,Conflict In Yemen,YEM001,Yem,Middle east,Complex crisis,Complex crisis,Yes,Individual,NaN,NaN,NaN
127,Yemen,Mixed Migration Flows In Yemen,YEM002,Yem,Middle east,International displacement,Mixed Migration,No,Individual,NaN,NaN,NaN
128,Zambia,Drought In Zambia,ZMB002,Zmb,Africa,Drought,Drought,Yes,Individual,NaN,NaN,NaN


In [62]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df11, on_col="Crisis Id")
merged_df

Number of rows deleted: 0
Indices of deleted duplicate rows: []
DataFrame after removing fully duplicate rows:
                                      Crisis            Type Crisis Id  \
0              Complex crisis in Afghanistan  Complex crisis    AFG001   
1       Nagorno-Karabakh Conflict in Armenia        Conflict    ARM002   
2    Nagorno-Karabakh conflict in Azerbaijan        Conflict    AZE002   
3                         Complex in Burundi  Complex crisis    BDI001   
4                   Conflict in Burkina Faso        Conflict    BFA002   
..                                       ...             ...       ...   
248                                      NaN             NaN       NaN   
249                                      NaN             NaN       NaN   
250                                      NaN             NaN       NaN   
251                                      NaN             NaN       NaN   
252                                      NaN             NaN       NaN   



,Crisis,Type,Crisis Id,Country,Iso3,Empowerment,Bti - Democracy Status,Trust In Society,Ethnic Fractionalisation,Size Of Excluded Ethnic Groups,Gender Inequality,Income Gini Coefficient,Inequality,Social Cohesion,Conflict Intensity,Total Killed In All Crisis,Safety And Security,Corruption Perception,Rule Of Law (Wgi),Rule Of Law (Bti),Freedom In The World,Rule Of Law,Society And Safety,# Of Different Types Of Affected Population Groups,Diversity Of Groups Affected,Impediments To Entry Into Country (Bureaucratic And Administrative),Restriction Of Movement (Impediments To Freedom Of Movement And/Or Administrative Restrictions),Interference Into Implementation Of Humanitarian Activities,"Violence Against Personnel, Facilities And Assets",Access Of Humanitarian Actors To Affected Populations,Denial Of Existence Of Humanitarian Needs Or Entitlements To Assistance,Restriction And Obstruction Of Access To Services And Assistance,Access Of People In Need To Aid,Ongoing Insecurity/Hostilities Affecting Humanitarian Assistance,Presence Of Mines And Improvised Explosive Devices,"Physical Constraints In The Environment (Obstacles Related To Terrain, Climate, Lack Of Infrastructure, Etc.)",Physical And Security Constraints,Humanitarian Access,Operating Environment,Type Of Crisis,# Of People In None/Minimal Conditions - Level 1,# Of People In Stressed Conditions - Level 2,# Of People In Moderate Conditions - Level 3,# Of People Severe Conditions - Level 4,# Of People Extreme Conditions - Level 5,People In Need,% Of People In None/Minimal Conditions - Level 1,% Of People In Stressed Conditions - Level 2,% Of People In Moderate Conditions - Level 3,% Of People Severe Conditions - Level 4,% Of People Extreme Conditions - Level 5,Concentration Of Conditions,Conditions Of People Affected,Total Population [Helper 0],Total Population [Figure],Total Population [Source],Total Population [Reliability],Total Population [Reliability Score],Total Population [Justification],Total Population [Link],Total Population [Date],Landmass Affected [Helper 1],Landmass Affected [Figure],Landmass Affected [Source],Landmass Affected [Reliability],Landmass Affected [Reliability Score],Landmass Affected [Justification],Landmass Affected [Link],Landmass Affected [Date],People Exposed [Helper 2],People Exposed [Figure],People Exposed [Source],People Exposed [Reliability],People Exposed [Reliability Score],People Exposed [Justification],People Exposed [Link],People Exposed [Date],People Affected [Helper 3],People Affected [Figure],People Affected [Source],People Affected [Reliability],People Affected [Reliability Score],People Affected [Justification],People Affected [Link],People Affected [Date],People Displaced [Helper 4],People Displaced [Figure],People Displaced [Source],People Displaced [Reliability],People Displaced [Reliability Score],People Displaced [Justification],People Displaced [Link],People Displaced [Date],Injuries Reported [Helper 5],Injuries Reported [Figure],Injuries Reported [Source],Injuries Reported [Reliability],Injuries Reported [Reliability Score],Injuries Reported [Justification],Injuries Reported [Link],Injuries Reported [Date],Illness Cases Reported [Helper 6],Illness Cases Reported [Figure],Illness Cases Reported [Source],Illness Cases Reported [Reliability],Illness Cases Reported [Reliability Score],Illness Cases Reported [Justification],Illness Cases Reported [Link],Illness Cases Reported [Date],Fatalities Reported [Helper 7],Fatalities Reported [Figure],Fatalities Reported [Source],Fatalities Reported [Reliability],Fatalities Reported [Reliability Score],Fatalities Reported [Justification],Fatalities Reported [Link],Fatalities Reported [Date],Buildings Damaged [Helper 8],Buildings Damaged [Figure],Buildings Damaged [Source],Buildings Damaged [Reliability],Buildings Damaged [Reliability Score],Buildings Damaged [Justification],Buildings Damaged [Link],Buildings Damaged [Date],Buildings Destroyed [Helper 9],Buildings Destroyed [Figure],Buildin

df12. Log.csv

In [63]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

Opening file: october-2021-Regional_Crises.csv
Current file index is: 17


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,CRISIS,CRISIS ID,COUNTRY,Iso3,Total sq.km.,Total population,Ethnic Fractionalisation Index,Empowerment Rights Index,size of excluded ethnic groups,BTI - Democracy Status,Gender Inequality Index,Income Gini coefficient,Conflict Intensity (HIIK),People killed in all crises,Rule of Law (WGI),Rule of Law (BTI),Freedom in the World Index,CPI
1,NaN,NaN,NaN,NaN,Index,Index,%,Index,Index,Index,Index,Number,Index,Index,Index,Index,Number,sq. Km
2,Regional Boko Haram Crisis,REG001,"Nigeria, Niger, Chad, Cameroon","NGA, NER, TCD, CMR",3909380,267437000,0.8092625009250001,6.25,0.075,4.508333333324999,0.6381952533333334,39.825,8.5,4189,-0.9582950472749999,3.9375,32.5,25.75
3,Venezuela Regional Crisis,REG002,"Venezuela, Brazil, Colombia, Ecuador, Peru, Tr...","VEN, BRA, COL, ECU, PER, TTO",11883180,338382000,0.4789911613333333,9.833333333333334,0.26604999999999995,6.563888888883333,0.3912139406833333,46.166666666666664,6,x,-0.6837853354666666,5.916666666666667,62.666666666666664,33.666666666666664
4,Regional Kashmir conflict,REG003,"India, Pakistan","IND, PAK",3744070,1561970000,0.7580873794499999,5,0.0835,5.5,0.52410904215,33.650000000000006,6,134,-0.34916854835,5.125,54.5,36.5
5,Syrian Regional Crisis,REG004,"Syria, Turkey, Iraq, Egypt, Jordan, Lebanon","SYR, TUR, IRQ, EGY, JOR, LBN",2482040,260081000,0.5096342366,3.5,0.32666666666666666,3.966666666683333,0.44548120716666667,34.31666666666667,8.333333333333334,3043,-0.8694431111166668,3.4583333333333335,27.5,30.5
6,Eastern Mediterranean Route,REG006,"Turkey, Greece","TUR, GRC",898530,95232000,0.2414492393,8,0.18350000000000002,4.9166666667,0.2136633259,37.4,8,26,-0.04206264765000001,3.5,60,43.5
7,Central Mediterranean Route,REG007,"Algeria, Tunisia, Libya, Italy","DZA, TUN, LBY, ITA",4590781,121936000,0.210782986625,5.75,0.068875,4.566666666666666,0.2460642436,32.1,5.5,604,-0.580277027575,4.166666666666667,50.5,37.25
8,Western Mediterranean Route,REG008,"Algeria, Morocco, Spain","DZA, MAR, ESP",3328251,125787000,0.46699264916666666,6.333333333333333,0.324,4.19166666665,0.33649607749999993,33.93333333333334,5.333333333333333,173,0.009313091633333334,3.75,54.333333333333336,46
9,Rohingya Regional Crisis,REG011,"Bangladesh, Myanmar","BGD, MMR",783250,194566000,0.35558001200000006,3.5,0.2125,3.85833333335,0.49717238359999993,31.549999999999997,7,126,-0.84954610465,3.125,34.5,27.5


In [64]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Log.csv"):
    df = clean_dataframe(df, crisis_id_col = 1)
    df12 = df
else: 
    print("Wrong file")
    current_file_index -= 1

Wrong file


In [ ]:
# df12

In [ ]:
#####-MERGE-#####

# merged_df = merge_df(merged_df, df12, on_col="Crisis Id")
merged_df

df13. Regional_Crises.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Regional_Crises.csv"):
    for col in range(5, len(df.columns)):
        if pd.notna(df.loc[1, col]):
            df.loc[0, col] =f"{df.loc[0, col]} [{df.loc[1, col]}]" 
    df = clean_dataframe(df, crisis_id_col = 1)
    df13 = df
else: print("Wrong file")

In [ ]:
df13

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df13, on_col="Crisis Id")
merged_df

df14. Reliability.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Reliability.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df14 = df
else: print("Wrong file")

In [ ]:
df14

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df14, on_col="Crisis Id")
merged_df

df15. Reliability_updated.csv.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Reliability_updated.csv"):
    df = clean_dataframe(df, crisis_id_col = 0)
    df15 = df
else: print("Wrong file")


In [ ]:
df15

In [ ]:
#####-MERGE-#####

merged_df = merge_df(merged_df, df15, on_col="Crisis Id")
merged_df

df16. Trends.csv

In [ ]:
df, current_file_index = load_next_csv(file_list, folder_path, current_file_index)
print(f"Current file index is: {current_file_index}")
df

In [ ]:
# Case-Specific

current_file = file_list[current_file_index]
if current_file.endswith("Trends.csv"):
    df = clean_dataframe(df, crisis_id_col = 2)
    df16 = df
else: print("Wrong file")


In [ ]:
df16

In [ ]:
#####-MERGE-#####

# merged_df = merge_df(merged_df, df16, on_col="Crisis Id")
merged_df

In [ ]:
# Remove rows where there are NaNs in the Crisis Id column and headers with NaNs or Unnamed
# merged_df =  drop_nan(merged_df, column_name="Crisis Id")

In [ ]:
# Add the year_month column as the first column
merged_df.insert(0, 'YYYY_MM', year_month)

# Reorder columns to have Crisis ID as the second column
columns = ['YYYY_MM', 'Crisis Id'] + [col for col in merged_df.columns if col not in ['YYYY_MM', 'Crisis Id']]
merged_df = merged_df[columns]

merged_df


In [ ]:
merged_df.to_csv(f"{year_month}_merged.csv", index=False)

In [ ]:
print(year_month)

In [ ]:
final_df = pd.read_csv(f"{year_month}_merged.csv")
final_df.shape

In [ ]:
print(final_df.name)